In [5]:
# 1. Cài đặt các thư viện cần thiết
!pip install Flask flask-ngrok pyngrok

In [7]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from flask import Flask, render_template, request, jsonify
from PIL import Image
import io

app = Flask(__name__)

# 1. KIẾN TRÚC MÔ HÌNH: Phải thay thế bằng Class bạn đã dùng khi Train
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 2. LOAD TRỌNG SỐ
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN(num_classes=10)

if os.path.exists('model.pt'):
    print("Đang load trọng số từ file model.pt...")
    model.load_state_dict(torch.load('model.pt', map_location=device))
else:
    print("CẢNH BÁO: Không tìm thấy file model.pt. Mô hình đang dùng trọng số ngẫu nhiên.")

model.to(device)
model.eval()

# 3. TIỀN XỬ LÝ
def transform_image(image_bytes):
    my_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    return my_transforms(image).unsqueeze(0).to(device)

# 4. NHÃN
labels = ['máy bay', 'ô tô', 'chim', 'mèo', 'nai', 'chó', 'ếch', 'ngựa', 'tàu', 'xe tải']

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        if 'file' not in request.files: return "No file"
        file = request.files['file']
        if file.filename == '': return "No selected file"
        img_bytes = file.read()
        tensor = transform_image(img_bytes)
        with torch.no_grad():
            outputs = model(tensor)
            _, predicted = torch.max(outputs, 1)
        result = labels[predicted.item()]
        return f"Kết quả dự đoán: <b>{result}</b> <br><br><a href='/'>Quay lại</a>"

    return '''
    <!doctype html>
    <title>CNN Classifier</title>
    <style>body { font-family: sans-serif; text-align: center; margin-top: 50px; }</style>
    <h1>Tải ảnh lên để phân loại (CIFAR-10)</h1>
    <form method=post enctype=multipart/form-data>
      <input type=file name=file>
      <input type=submit value=Upload>
    </form>
    '''

CẢNH BÁO: Không tìm thấy file model.pt. Mô hình đang dùng trọng số ngẫu nhiên.


In [ ]:
# 4. Chạy ứng dụng với ngrok (Dành riêng cho Colab)
from pyngrok import ngrok
import os

# Đảm bảo app đã được import nếu chạy ở cell khác
try:
    from __main__ import app
except ImportError:
    pass

# BẮT BUỘC: Thay 'YOUR_AUTHTOKEN_HERE' bằng token lấy từ https://dashboard.ngrok.com/get-started/your-authtoken
AUTH_TOKEN = "3Cr83gp6l0Zg8pbiQkNlGowdtvz_To3u9E2oGPBV51oEsDS1"

if AUTH_TOKEN == "YOUR_AUTHTOKEN_HERE":
    print("Vui lòng nhập Authtoken của bạn vào biến AUTH_TOKEN!")
else:
    try:
        # Cấu hình token
        ngrok.set_auth_token(AUTH_TOKEN)

        # Hủy các tunnel cũ nếu có để tránh lỗi
        tunnels = ngrok.get_tunnels()
        for tunnel in tunnels:
            ngrok.disconnect(tunnel.public_url)

        # Mở tunnel mới
        public_url = ngrok.connect(5000)
        print("\n" + "="*50)
        print(f"LINK TRUY CẬP WEBSITE: {public_url.public_url}")
        print("="*50 + "\n")

        # Chạy Flask app
        # Lưu ý: Nếu vẫn báo lỗi 'app' not defined, hãy chạy lại Cell f0b1dd13 trước.
        app.run(port=5000, debug=False, use_reloader=False)
    except NameError:
        print("LỖI: Biến 'app' chưa được định nghĩa. Vui lòng chạy cell chứa code Flask (cell f0b1dd13) trước khi chạy cell này!")
    except Exception as e:
        print(f"Lỗi kết nối: {e}")


LINK TRUY CẬP WEBSITE: https://snowdrop-paging-riptide.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 16:07:40] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 16:07:51] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Apr/2026 16:07:55] "GET / HTTP/1.1" 200 -
